In [1]:
import napari
import numpy as np
import pandas as pd
import re
from pathlib import Path
import magicgui
import zipfile
from qtpy.QtWidgets import QSizePolicy
from napari_orthogonal_views.ortho_view_manager import show_orthogonal_views
%matplotlib qt

In [2]:
def get_clipping_plane_vals(layer):
    x1, x2, y1, y2, z1, z2 = [val.position for val in layer.experimental_clipping_planes]
    x_range = (x1[0],x2[0])
    y_range = (y1[1],y2[1])
    z_range = (z1[2],z2[2])
    return x_range, y_range, z_range

In [3]:
# green_zarr_path=input("Path to green channel Zarr: ")
# red_zarr_path=input("Path to red channel Zarr: ")
# SN_zip=input("Path to zip file: ")

# green_zarr_path= str(Path(green_zarr_path))
# red_zarr_path= str(Path(red_zarr_path))
# SN_zip= str(Path(SN_zip))

In [4]:
# New CND-1
green_zarr_path= str(Path(r"C:\Users\yarbroughb\OneDrive - Howard Hughes Medical Institute\shroff\Lineaging Napari 3D viewer\green_channel.zarr"))
red_zarr_path= str(Path(r"C:\Users\yarbroughb\OneDrive - Howard Hughes Medical Institute\shroff\Lineaging Napari 3D viewer\red_channel.zarr"))
SN_zip= str(Path(r"Z:\shrofflab\CND-1_RedUntwisting_A\Lineaging\20260225\20260225_CND-1_lineage\Pos0\SPIMB\For_Deep_Learning\For_Lineaging\StarryNite\SN_files\Decon_emb1_edited.zip"))

In [5]:
# # Old CND-1
# green_zarr_path= str(Path(r"Z:\shrofflab\CND-1_RedUntwisting_A\Lineaging\041423\CND-1_BV514_Overnight\BV514_CND1\Pos1\SPIMB\Reg_Sample\For_Lineaging\StarryNite\green_channel.zarr"))
# red_zarr_path= str(Path(r"Z:\shrofflab\CND-1_RedUntwisting_A\Lineaging\041423\CND-1_BV514_Overnight\BV514_CND1\Pos1\SPIMB\Reg_Sample\For_Lineaging\StarryNite\red_channel.zarr"))
# SN_zip= str(Path(r"Z:\shrofflab\CND-1_RedUntwisting_A\Lineaging\041423\CND-1_BV514_Overnight\BV514_CND1\Pos1\SPIMB\Reg_Sample\For_Lineaging\StarryNite\SN_files\Decon_emb1_edited.zip"))

In [6]:
# #JJ
# green_zarr_path= str(Path(r"X:\\shrofflab\\OH15257\\Lineaging\\20260313\\BV514_x_OH15257\\Pos1\\SPIMB\\For_Deep_Learning\\For_Lineaging\\StarryNite\\green_channel.zarr"))
# red_zarr_path= str(Path(r"X:\\shrofflab\\OH15257\\Lineaging\\20260313\\BV514_x_OH15257\\Pos1\\SPIMB\\For_Deep_Learning\\For_Lineaging\\StarryNite\\red_channel.zarr"))
# SN_zip= str(Path(r"X:\\shrofflab\\OH15257\\Lineaging\\20260313\\BV514_x_OH15257\\Pos1\\SPIMB\\For_Deep_Learning\\For_Lineaging\\StarryNite\\SN_files\\Decon_emb1_edited_v2.zip"))

In [5]:
nuclei_coords = list()
nuclei_labels = list()
nuclei_ids = dict()

i=0
with zipfile.ZipFile(SN_zip, 'r') as zf:
    files = [file for file in zf.namelist() if file.endswith('nuclei')]

    for file in files:
        with zf.open(file) as f:

            idx = re.search(r'\d{3}',file)
            idx = int(idx.group())-1
            
            raw = f.read().decode('utf-8')
            raw = raw.split(' \n')
            raw = [x.split(', ') for x in raw]

            # FORMAT AND EXTRACT COORDINATE INFORMATION FROM NUCLEI FILES
            content = pd.DataFrame(raw).dropna(axis=0,thresh=4)
            content = content[content[9]!='']
            content = content[[7,6,5,9]]
            content.set_index(9, inplace=True)
            coordinate_time_idx = np.array([[idx]]).repeat(len(content),axis=0)
            content['t'] = coordinate_time_idx
            content = content[['t',7,6,5]]

            # BUILD DICT OF ALL NUCLEI PRESENT ACCROSS TIMESERIES
            labels = content.index.tolist()
            for x in labels:
                if x not in nuclei_ids.values():
                    nuclei_ids.setdefault(i,x)
                    i+=1

            # MAP SULSTON NAMES TO INTEGER KEYS
            content.index = content.index.map({value: int(key) for key, value in nuclei_ids.items()})
            coords = np.array(content.reset_index())
            nuclei_coords.append(coords)
            nuclei_labels.append(labels)

nuclei_coords = np.vstack(nuclei_coords)
nuclei_coords = nuclei_coords.astype(np.float64)
nuclei_tracks = nuclei_coords[np.lexsort((nuclei_coords[:,1],nuclei_coords[:,0]))]
nuclei_points = nuclei_coords[:,1:]
nuclei_labels = np.concat((nuclei_labels))

In [6]:
Δx = list()
Δy = list()
Δz = list()

for nuc_id in nuclei_ids.keys():
    coords = nuclei_tracks[nuclei_tracks[:,0]==nuc_id][:,2:]    
    diff = np.diff(coords,axis=0,prepend=np.array([coords[0,:]]))
    
    Δx.append(diff[:,2])
    Δy.append(diff[:,1])
    Δz.append(diff[:,0])

Δx = np.concat(Δx)
Δy = np.concat(Δy)
Δz = np.concat(Δz)

In [7]:
track_features={
    'sulston_name': [nuclei_ids[x] for x in nuclei_tracks[:,0]],
    'x': nuclei_tracks[:,4],
    'y': nuclei_tracks[:,3],
    'z': nuclei_tracks[:,2],
    't': nuclei_tracks[:,1],
    'Δx':Δx,
    'Δy':Δy,
    'Δz':Δz,
}

point_properties = {
    'sulston_name': [nuclei_ids[x] for x in nuclei_coords[:,0]],
    'x': nuclei_tracks[:,4],
    'y': nuclei_tracks[:,3],
    'z': nuclei_tracks[:,2],
    't': nuclei_tracks[:,1],
    'Δx':Δx,
    'Δy':Δy,
    'Δz':Δz
}

text = {
    'string': '{sulston_name}',
    'size': 10,
    'color': 'cyan',
    'translation': np.array([0, -2, -2, -2]),
    'anchor': 'upper_left'
}

In [50]:
viewer = napari.Viewer()
# show_orthogonal_views(viewer)
viewer.dims.ndisplay = 3

viewer.open(
    path=green_zarr_path,
    blending='additive',
    contrast_limits = (0,300),
    colormap='green',
    rendering='attenuated_mip',
    attenuation=0.75,
           )

viewer.open(
    path=red_zarr_path,
    blending='additive',
    contrast_limits = (0,300),
    colormap='red',
    rendering='attenuated_mip',
    attenuation=0.75,
           )

viewer.add_tracks(
    nuclei_tracks,
    features=track_features,
    colormap='viridis',
    tail_length=2
)

viewer.add_points(
    nuclei_points,
    ndim=4,
    opacity=1,
    size=3,
    face_color='cyan',
    border_color='cyan',
    properties=point_properties,
    text=text,
    blending='additive'
)
    
green_layer = viewer.layers[0]
red_layer = viewer.layers[1]
track_layer = viewer.layers[2]
point_layer = viewer.layers[3]

layer_data = green_layer.data
t, x, y, z = layer_data.shape

green_layer.experimental_clipping_planes=[
        {'position': (0, 0, 0), 'normal': (1, 0, 0), 'enabled': True},
        {'position': (x, 0, 0), 'normal': (-1, 0, 0), 'enabled': True},
        {'position': (0, y, 0), 'normal': (0, -1, 0), 'enabled': True},
        {'position': (0, 0, 0), 'normal': (0, 1, 0), 'enabled': True},
        {'position': (0, 0, 0), 'normal': (0, 0, 1), 'enabled': True},
        {'position': (0, 0, z), 'normal': (0, 0, -1), 'enabled': True},
    ]

red_layer.experimental_clipping_planes=[
        {'position': (0, 0, 0), 'normal': (1, 0, 0), 'enabled': True},
        {'position': (x, 0, 0), 'normal': (-1, 0, 0), 'enabled': True},
        {'position': (0, y, 0), 'normal': (0, -1, 0), 'enabled': True},
        {'position': (0, 0, 0), 'normal': (0, 1, 0), 'enabled': True},
        {'position': (0, 0, 0), 'normal': (0, 0, 1), 'enabled': True},
        {'position': (0, 0, z), 'normal': (0, 0, -1), 'enabled': True},
    ]

track_layer.experimental_clipping_planes=[
        {'position': (0, 0, 0), 'normal': (1, 0, 0), 'enabled': True},
        {'position': (x, 0, 0), 'normal': (-1, 0, 0), 'enabled': True},
        {'position': (0, y, 0), 'normal': (0, -1, 0), 'enabled': True},
        {'position': (0, 0, 0), 'normal': (0, 1, 0), 'enabled': True},
        {'position': (0, 0, 0), 'normal': (0, 0, 1), 'enabled': True},
        {'position': (0, 0, z), 'normal': (0, 0, -1), 'enabled': True},
    ]

point_layer.experimental_clipping_planes=[
        {'position': (0, 0, 0), 'normal': (1, 0, 0), 'enabled': True},
        {'position': (x, 0, 0), 'normal': (-1, 0, 0), 'enabled': True},
        {'position': (0, y, 0), 'normal': (0, -1, 0), 'enabled': True},
        {'position': (0, 0, 0), 'normal': (0, 1, 0), 'enabled': True},
        {'position': (0, 0, 0), 'normal': (0, 0, 1), 'enabled': True},
        {'position': (0, 0, z), 'normal': (0, 0, -1), 'enabled': True},
    ]

# CLIPPING PLANE SLIDER FOR X
@magicgui.magicgui(
    auto_call=True,
    threshold={'widget_type': 'RangeSlider', 'min': 0, 'max': x, 'label': 'Clip x', 'orientation': 'vertical'},
)

def clip_x(threshold=(0,x)):

    green_layer.experimental_clipping_planes[0].position = (threshold[0], 0, 0)
    green_layer.experimental_clipping_planes[1].position = (threshold[1], 0, 0)
    
    red_layer.experimental_clipping_planes[0].position = (threshold[0], 0, 0)
    red_layer.experimental_clipping_planes[1].position = (threshold[1], 0, 0)
    
    track_layer.experimental_clipping_planes[0].position = (threshold[0], 0, 0)
    track_layer.experimental_clipping_planes[1].position = (threshold[1], 0, 0)
    
    point_layer.experimental_clipping_planes[0].position = (threshold[0], 0, 0)
    point_layer.experimental_clipping_planes[1].position = (threshold[1], 0, 0)
    
    return green_layer.experimental_clipping_planes, red_layer.experimental_clipping_planes, track_layer.experimental_clipping_planes, point_layer.experimental_clipping_planes

clip_x.min_width = 200
clip_x.min_height = 200
clip_x.max_height = 600
clip_x.native.setSizePolicy(
    QSizePolicy.Policy.Expanding,
    QSizePolicy.Policy.Expanding,
)

# CLIPPING PLANE SLIDER FOR Y
@magicgui.magicgui(
    auto_call=True,
    threshold={'widget_type': 'RangeSlider', 'min': 0, 'max': y, 'label': 'Clip y','orientation': 'vertical'},
)

def clip_y(threshold=(0,y)):

    green_layer.experimental_clipping_planes[2].position = ( 0, threshold[1], 0)
    green_layer.experimental_clipping_planes[3].position = ( 0, threshold[0], 0)

    red_layer.experimental_clipping_planes[2].position = ( 0, threshold[1], 0)
    red_layer.experimental_clipping_planes[3].position = ( 0, threshold[0], 0)

    track_layer.experimental_clipping_planes[2].position = ( 0, threshold[1], 0)
    track_layer.experimental_clipping_planes[3].position = ( 0, threshold[0], 0)

    point_layer.experimental_clipping_planes[2].position = ( 0, threshold[1], 0)
    point_layer.experimental_clipping_planes[3].position = ( 0, threshold[0], 0)

    return green_layer.experimental_clipping_planes, red_layer.experimental_clipping_planes, track_layer.experimental_clipping_planes, point_layer.experimental_clipping_planes

clip_y.min_width = 200
clip_y.min_height = 200
clip_y.max_height = 600
clip_y.native.setSizePolicy(
    QSizePolicy.Policy.Expanding,
    QSizePolicy.Policy.Expanding,
)

# CLIPPING PLANE SLIDER FOR Z
@magicgui.magicgui(
    auto_call=True,
    threshold={'widget_type': 'RangeSlider', 'min': 0, 'max': z, 'label': 'Clip z','orientation': 'horizontal'},
)

def clip_z(threshold=(0,z)):

    green_layer.experimental_clipping_planes[4].position = ( 0,  0, threshold[0])
    green_layer.experimental_clipping_planes[5].position = ( 0,  0, threshold[1])

    red_layer.experimental_clipping_planes[4].position = ( 0,  0, threshold[0])
    red_layer.experimental_clipping_planes[5].position = ( 0,  0, threshold[1])

    track_layer.experimental_clipping_planes[4].position = ( 0,  0, threshold[0])
    track_layer.experimental_clipping_planes[5].position = ( 0,  0, threshold[1])

    point_layer.experimental_clipping_planes[4].position = ( 0,  0, threshold[0])
    point_layer.experimental_clipping_planes[5].position = ( 0,  0, threshold[1])

    return green_layer.experimental_clipping_planes, red_layer.experimental_clipping_planes, track_layer.experimental_clipping_planes, point_layer.experimental_clipping_planes

viewer.window.add_dock_widget(clip_x, name='Clipping Planes X', area='right')
viewer.window.add_dock_widget(clip_y, name='Clipping Planes Y', area='right')
viewer.window.add_dock_widget(clip_z, name='Clipping Planes Z', area='bottom')
# features_table=viewer.window.add_plugin_dock_widget('napari', 'Features table widget')
# features_table[0].area = 'left'

napari.run()

In [33]:
"""
Code to add lineage view to the existing napari viewer with StarryNite tracking data.

This should be added to the Viewer.ipynb notebook after the existing visualization code.
The nuclei_coords and nuclei_labels variables should already be loaded from
the StarryNite data.
"""

import networkx as nx
from funtracks.data_model import SolutionTracks
from motile_tracker.data_views.views.tree_view.tree_widget import TreeWidget
from motile_tracker.data_views.views_coordinator.tracks_viewer import TracksViewer

# Build networkx graph from StarryNite data
# nuclei_coords format: [[t, z, y, x], ...]
# nuclei_labels format: ['cell_name1', 'cell_name2', ...]

print("Building graph from StarryNite data...")

# Create a mapping from (time, name) to index for quick lookup
time_name_to_idx = {}
for idx, (coords, label) in enumerate(zip(nuclei_coords, nuclei_labels, strict=True)):
    t = int(coords[1])
    time_name_to_idx[(t, label)] = idx

# Build networkx directed graph
G = nx.DiGraph()

# Add nodes with their attributes
for idx, (coords, label) in enumerate(
    zip(nuclei_coords, nuclei_labels, strict=True)
):
    t = int(coords[1])
    z, y, x = coords[2], coords[3], coords[4]

    G.add_node(
        idx,
        time=t,
        z=z,
        y=y,
        x=x,
        name=label,
    )

Building graph from StarryNite data...


In [34]:
G.nodes[0]

{'time': 0,
 'z': np.float64(93.0),
 'y': np.float64(209.0),
 'x': np.float64(206.0),
 'name': np.str_('AB')}

In [51]:
# Hard-coded division rules for known C. elegans naming exceptions
# Format: parent_name -> [daughter1_name, daughter2_name, ...]
DIVISION_RULES = {
    "P0": ["AB", "P1"],
    "P1": ["EMS", "P2"],
    "P2": ["C", "P3"],
    "P3": ["D", "P4"],
    "EMS": ["MS", "E"],
    # Add more known exceptions here as needed
}


def get_daughters(parent_name, candidates_at_next_t):
    """Get daughter cell names for a parent cell.

    First checks hard-coded rules, then falls back to prefix-based detection.

    Args:
        parent_name: Name of the parent cell
        candidates_at_next_t: List of (name, idx) tuples for cells at next timepoint

    Returns:
        List of indices of daughter cells
    """
    candidate_names = {name: idx for name, idx in candidates_at_next_t}

    # Check hard-coded rules first
    if parent_name in DIVISION_RULES:
        daughters = []
        for daughter_name in DIVISION_RULES[parent_name]:
            if daughter_name in candidate_names:
                daughters.append(candidate_names[daughter_name])
        if daughters:  # Found at least one daughter from rules
            return daughters

    # Fall back to prefix-based detection
    # Look for cells where parent name is a prefix and length is parent + 1
    daughters = []
    for candidate_name, candidate_idx in candidates_at_next_t:
        if (
            candidate_name.startswith(parent_name)
            and len(candidate_name) == len(parent_name) + 1
        ):
            daughters.append(candidate_idx)

    return daughters


# Add edges between cells across timepoints
# Handle both:
# 1. Same cell continuing (same name)
# 2. Cell divisions (using hard-coded rules + prefix detection)
print("Adding edges between adjacent timepoints...")
edge_count = 0
division_count = 0

for idx, (coords, label) in enumerate(
    zip(nuclei_coords, nuclei_labels, strict=True)
):
    t = int(coords[1])
    next_t = t + 1

    # Case 1: Same cell continues to next timepoint (no division)
    next_key = (next_t, label)
    if next_key in time_name_to_idx:
        next_idx = time_name_to_idx[next_key]
        G.add_edge(idx, next_idx)
        edge_count += 1
    else:
        # Case 2: Cell divides - find daughters at next timepoint
        candidates = [
            (name, idx)
            for (check_t, name), idx in time_name_to_idx.items()
            if check_t == next_t
        ]

        daughters = get_daughters(label, candidates)

        # Add edges to all daughters
        for daughter_idx in daughters:
            G.add_edge(idx, daughter_idx)
            edge_count += 1
            division_count += 1

print(
    f"Created graph with {G.number_of_nodes()} nodes and {edge_count} edges "
    f"({division_count} division edges)"
)

Adding edges between adjacent timepoints...
Created graph with 14856 nodes and 14790 edges (316 division edges)


In [52]:
# Create SolutionTracks object (release version uses NetworkX directly)
print("Creating SolutionTracks object...")
tracks = SolutionTracks(
    graph=G,
    time_attr="time",
    pos_attr=["z", "y", "x"],  # Use individual attributes for each axis
    ndim=4,  # 3D + time
    scale=[1.0, 1.0, 1.0, 1.0],  # Adjust if needed
)

print(f"SolutionTracks created with {len(tracks.graph.nodes)} nodes")

# Set up the TracksViewer and add tracks
print("Setting up TracksViewer...")
tracks_viewer = TracksViewer.get_instance(viewer)
tracks_viewer.update_tracks(tracks, name="StarryNite Tracks")

# Add cell names as properties to the points layer for display
print("Adding cell names to track points...")
points_layer = None
for layer in viewer.layers:
    # Look for the points layer (ends with "_points")
    if layer.name.endswith("Tracks_points"):
        points_layer = layer
        break

if points_layer is not None:
    # Get cell names for all nodes in order
    node_ids = points_layer.features["node_id"]
    cell_names = [tracks.graph.nodes[node_id].get("name", "") for node_id in node_ids]

    # Add names to features (not properties - features is the editable DataFrame)
    points_layer.features["name"] = cell_names

    # Enable text display with cell names
    points_layer.text = "name"
    points_layer.text.translation = [0, -2, -2, -2]
    points_layer.text.anchor = "upper_left"

    # Refresh the layer to update the display
    points_layer.refresh()

    print(f"✓ Added {len(cell_names)} cell names to track points layer")
    print(f"  Sample names: {cell_names[:5]}")
else:
    print("Warning: Could not find track points layer")

# Add the lineage tree widget to the viewer
print("Adding lineage view widget...")
tree_widget = TreeWidget(viewer)
viewer.window.add_dock_widget(tree_widget, name="Lineage View", area="bottom")

print("✓ Lineage view added successfully!")

Creating SolutionTracks object...
SolutionTracks created with 14856 nodes
Setting up TracksViewer...
Adding cell names to track points...
✓ Added 14856 cell names to track points layer
  Sample names: [np.str_('AB'), np.str_('P1'), np.str_('Nuc1'), np.str_('Nuc2'), np.str_('AB')]
Adding lineage view widget...
✓ Lineage view added successfully!


C:\Users\yarbroughb\AppData\Local\miniconda3\envs\shroff-python-313\Lib\site-packages\napari\layers\utils\style_encoding.py:252: RuntimeWarning: Applying the encoding failed. Using the safe fallback value instead.
  warnings.warn(
C:\Users\yarbroughb\AppData\Local\miniconda3\envs\shroff-python-313\Lib\site-packages\napari\layers\utils\style_encoding.py:252: RuntimeWarning: Applying the encoding failed. Using the safe fallback value instead.
  warnings.warn(
C:\Users\yarbroughb\AppData\Local\miniconda3\envs\shroff-python-313\Lib\site-packages\napari\layers\utils\style_encoding.py:252: RuntimeWarning: Applying the encoding failed. Using the safe fallback value instead.
  warnings.warn(
C:\Users\yarbroughb\AppData\Local\miniconda3\envs\shroff-python-313\Lib\site-packages\napari\layers\utils\style_encoding.py:252: RuntimeWarning: Applying the encoding failed. Using the safe fallback value instead.
  warnings.warn(


In [37]:
tracks_viewer = TracksViewer.get_instance(viewer)

In [46]:
tracks_viewer.tracking_layers.tracks_layer.as_layer_data_tuple()

(array([[  1. ,   0. ,  93. , 209. , 206. ],
        [  1. ,   1. , 102. , 203. , 206. ],
        [  1. ,   2. , 102. , 204. , 209. ],
        ...,
        [381. , 419. ,  38. , 189. , 186. ],
        [381. , 420. ,  42.5, 189. , 183. ],
        [382. , 420. ,  27. , 104. ,  35. ]], shape=(14856, 5)),
 {'affine': array([[1., 0., 0., 0., 0.],
         [0., 1., 0., 0., 0.],
         [0., 0., 1., 0., 0.],
         [0., 0., 0., 1., 0.],
         [0., 0., 0., 0., 1.]]),
  'axis_labels': ('-4', '-3', '-2', '-1'),
  'blending': 'additive',
  'experimental_clipping_planes': [],
  'metadata': {},
  'name': 'StarryNite Tracks_tracks',
  'opacity': 1.0,
  'projection_mode': <BaseProjectionMode.NONE: 'none'>,
  'rotate': [[np.float64(1.0),
    np.float64(0.0),
    np.float64(0.0),
    np.float64(0.0)],
   [np.float64(0.0), np.float64(1.0), np.float64(0.0), np.float64(0.0)],
   [np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0)],
   [np.float64(0.0), np.float64(0.0), np.float64(0.0